In [1]:
from bs4 import BeautifulSoup
import requests
import unicodedata
import re
import math


In [2]:
LINK = "https://ic2s2-2025.org/program/"
r = requests.get(LINK)


In [3]:
soup = BeautifulSoup(r.content)

In [4]:
table = soup.find("table", {"class": "tutorials"})

In [5]:
table_rows = table.find_all("tr")
table_rows

[<tr>
 <th>Time</th>
 <th>Monday (July 21, 2025)</th>
 <th>Tuesday (July 22, 2025)</th>
 <th>Wednesday (July 23, 2025)</th>
 <th>Thursday (July 24, 2025)</th>
 </tr>,
 <tr>
 <th>7:30</th>
 <td>Registration desk opens</td>
 <td>Registration desk opens</td>
 <td>Registration desk opens</td>
 <td></td>
 </tr>,
 <tr>
 <th>8:45</th>
 <td></td>
 <td>Opening remarks</td>
 <td></td>
 <td></td>
 </tr>,
 <tr>
 <th>9:00</th>
 <td rowspan="3">Morning tutorials</td>
 <td>
 						Keynote: Dean Eckles<!--Keynote 1--><br/>
 						Keynote: Kathleen Carley<!--Keynote 2-->
 </td>
 <td>
 						Keynote: Laura Nelson<!--Keynote 5--><br/>
 						Lightning talks
 					</td>
 <td>
 						Keynote: Duncan J. Watts<!--Keynote 8--><br/>
 						Lightning talks
 					</td>
 </tr>,
 <tr>
 <th>10:30</th>
 <td colspan="3" style="text-align: center; font-weight: bold;">Coffee break</td>
 </tr>,
 <tr>
 <th>11:00</th>
 <td>Parallel sessions 1</td>
 <td>Parallel sessions 3</td>
 <td>Parallel sessions 5</td>
 </tr>,
 <tr>
 

### Program Link

In [6]:
keynotes = []
for row in table_rows:
    tds = row.find_all("td")
    for td in tds:
        for chunk in td.stripped_strings:
            if chunk.lower().startswith("keynote:"):
                name = chunk.split(":", 1)[1].strip()
                keynotes.append(name)

keynotes

['Dean Eckles',
 'Kathleen Carley',
 'Laura Nelson',
 'Duncan J. Watts',
 'Brandon Stewart',
 'Lisa P. Argyle',
 'Amir Goldberg',
 'Arnout van de Rijt',
 'Sarah Williams']

In [7]:
import pandas as pd
all_authors = []
df = pd.read_csv("https://ic2s2-2025.org/files/ic2s2_2025_schedule_v5.csv")
for i in range(len(df)):
    author_sec = (df["author"].loc[i])
    if pd.notna(author_sec):   
        author_sec_list = author_sec.split(", ")
        for author in author_sec_list:
            if author.lower().strip() not in all_authors:
                all_authors.append(author.lower().strip())



### Organization Link

In [17]:
link2 = "https://ic2s2-2025.org/organization/"
r2 = requests.get(link2)
soup2 = BeautifulSoup(r2.content)

In [9]:
sections_program_chairs = soup2.find_all("section", id="program-chairs")

for section in sections_program_chairs:
    chairs = section.find_all("h3")
    for chair in chairs:
        if chair.text.strip().lower() not in all_authors:
            all_authors.append(chair.text.strip().lower())


In [10]:
section_tutorial_chairs = soup2.find_all("section",id = "tutorial-chairs")

for section in section_tutorial_chairs:
    chairs = section.find_all("h3")
    for chair in chairs:
        if chair.text.strip().lower() not in all_authors:
            all_authors.append(chair.text.strip().lower())


In [11]:
section_social_media_chairs = soup2.find_all("section",id ="website-social-media-chairs")

for section in section_social_media_chairs:
    chairs = section.find_all("h3")
    for chair in chairs:
        if chair.text.strip().lower() not in all_authors:
            all_authors.append(chair.text.strip().lower())



In [12]:
section_volunteers = soup2.find_all("section",id ="conference-volunteers")

for section in section_volunteers:
    chairs = section.find_all("h3")
    for chair in chairs:
        if chair.text.strip().lower() not in all_authors:
            all_authors.append(chair.text.strip().lower())



In [13]:
section_general_chairs = soup2.find_all("section",id = "general-chairs")

for section in section_general_chairs:
    chairs = section.find_all("h3")
    for chair in chairs:
        if chair.text.strip().lower() not in all_authors:
            all_authors.append(chair.text.strip().lower())


### Finding a distance threshold between names to reduce duplicates based on spelling and accents

In [14]:
from rapidfuzz.distance import Levenshtein
low_distances = []
for i in range(len(all_authors)):
    for j in range(i + 1, len(all_authors)):
        if Levenshtein.distance(all_authors[i], all_authors[j]) < 5:
            low_distances.append((all_authors[i], all_authors[j]))
low_distances

[('étienne ollion', 'etienne ollion'),
 ('émilien schultz', 'emilien schultz'),
 ('caspar van lissa', 'caspar j. van lissa'),
 ('thomas li', 'thomas h. li'),
 ('thomas li', 'tomas ruiz'),
 ('chen zhong', 'yuan zhang'),
 ('chen zhong', 'ying zhong'),
 ('chen zhong', 'lechen zhang'),
 ('chen zhong', 'nan zhang'),
 ('chen zhong', 'yue zheng'),
 ('kristina gligoric', 'kristina gligorić'),
 ('cinoo lee', 'minjin lee'),
 ('cinoo lee', 'jungwoo lee'),
 ('mark whiting', 'mark e. whiting'),
 ('duncan watts', 'duncan j. watts'),
 ('johannes mast', 'johannes wachs'),
 ('kathleen carly', 'kathleen m. carley'),
 ('brandon stewart', 'brandon m. stewart'),
 ('laura nelson', 'laura diosan'),
 ('amir goldberg', 'amit goldenberg'),
 ('nay san', 'pu yan'),
 ('nay san', 'yao wan'),
 ('nay san', 'cai yang'),
 ('nay san', 'ana ma'),
 ('nay san', 'yan wang'),
 ('nay san', 'na jiang'),
 ('nay san', 'nan zhang'),
 ('nay san', 'may lim'),
 ('xuan zhao', 'yuan zhang'),
 ('xuan zhao', 'yuan liao'),
 ('xuan zhao',

### Cleaning the feasible duplicates

In [ ]:
def normalize_name(name):
    # Lowercase
    name = name.lower()
    
    # Normalize unicode (separate accents)
    name = unicodedata.normalize('NFKD', name)
    name = ''.join(c for c in name if not unicodedata.combining(c))
    
    # Standardize apostrophes
    name = name.replace("’", "'")
    
    # Remove dots
    name = name.replace(".", "")
    
    # Remove single-letter middle initials
    tokens = name.split()
    tokens = [t for t in tokens if len(t) > 1]
    
    # Rejoin
    name = " ".join(tokens)
    
    return name.strip()

def deduplicate_authors(author_list):
    seen = {}
    
    for name in author_list:
        normalized = normalize_name(name)
        
        # Keep the first version encountered
        if normalized not in seen:
            seen[normalized] = name
    
    return list(seen.values())
all_authors_clean = deduplicate_authors(all_authors)
all_authors_clean = [author.title() for author in all_authors_clean]


file = open('authors.txt','w')
for author in all_authors_clean:
	file.write(author+"\n")
file.close()
